# 03 - Forecasting

Benchmark four time-series models (ARIMA, Prophet, ETS, XGBoost) on weekly
PR counts, compare error metrics, and plot forecasts against actuals.

In [ ]:
from pathlib import Path

import pandas as pd

from oss_pulse.analyze.forecast import (
    benchmark_models,
    fit_arima,
    fit_ets,
    fit_prophet,
    fit_xgboost_ts,
)
from oss_pulse.visualize.timeseries import plot_forecast
from oss_pulse.visualize.style import setup_style

setup_style()

In [ ]:
# Load weekly data
DATA_DIR = Path("../data/processed")
weekly_df = pd.read_parquet(DATA_DIR / "repo_weekly.parquet")

# Pick the first repo with enough data points
target_repo = weekly_df["repo_name"].unique()[0]
repo_weekly = (
    weekly_df[weekly_df["repo_name"] == target_repo]
    .sort_values("year_week")
    .reset_index(drop=True)
)

series = repo_weekly.set_index("year_week")["pr_count"]
print(f"Repo: {target_repo}")
print(f"Series length: {len(series)} weeks")

In [ ]:
# Benchmark all four models
# TODO: run with real data
comparison = benchmark_models(series)
print("Model Comparison:")
comparison

In [ ]:
# Individual model forecast plots
# TODO: run with real data
split_idx = int(len(series) * 0.8)
train, test = series.iloc[:split_idx], series.iloc[split_idx:]

# ARIMA forecast
arima_result = fit_arima(series)
fig = plot_forecast(
    test,
    arima_result["predictions"],
    title=f"ARIMA Forecast: {target_repo} (MAE={arima_result['mae']:.2f})",
)
fig.show()

# ETS forecast
ets_result = fit_ets(series)
fig = plot_forecast(
    test,
    ets_result["predictions"],
    title=f"ETS Forecast: {target_repo} (MAE={ets_result['mae']:.2f})",
)
fig.show()

In [ ]:
# Prophet and XGBoost forecasts
# TODO: run with real data
prophet_df = pd.DataFrame({"ds": series.index, "y": series.values})

prophet_result = fit_prophet(prophet_df)
fig = plot_forecast(
    pd.Series(prophet_df["y"].values[split_idx:]),
    prophet_result["predictions"],
    title=f"Prophet Forecast: {target_repo} (MAE={prophet_result['mae']:.2f})",
)
fig.show()

xgb_result = fit_xgboost_ts(prophet_df)
fig = plot_forecast(
    pd.Series(prophet_df["y"].values[split_idx:]),
    xgb_result["predictions"],
    title=f"XGBoost Forecast: {target_repo} (MAE={xgb_result['mae']:.2f})",
)
fig.show()

In [ ]:
# Comparison bar chart
import matplotlib.pyplot as plt
from oss_pulse.visualize.style import PALETTE

fig, ax = plt.subplots(figsize=(10, 5))
comparison.set_index("model")[["mae", "rmse"]].plot.bar(
    ax=ax, color=[PALETTE["primary"], PALETTE["accent"]], edgecolor=PALETTE["bg"],
)
ax.set_title("Model Error Comparison", fontsize=14, fontweight="bold")
ax.set_ylabel("Error")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()